# Naive RAG with the NetCore Knowledge Base

**Naive RAG** (LangChain calls it *2-step RAG*) always **retrieves first, then generates**, using exactly one LLM call per question:

```
Question ──► Retriever ──► Top-k chunks ──► Context ──► Prompt ──► LLM ──► Grounded answer + citations
             (similarity | keyword | hybrid)        (system rules + context + question)
```

| Step | Where it lives |
|---|---|
| Vector databases | Chroma server: `networking_kb`, `meetings_kb` (built by `make populate`) |
| Retrieval functions | `scripts/rag_search.py`: `similarity_search`, `keyword_search`, `hybrid_search`, `get_retriever`, `format_docs` |
| Prompt + LLM | This notebook (`OPENAI_MODEL` from `.env`) |

**Before running:** `make chroma-up` (and `make populate` once), then select the project's `.venv` kernel.

## 0. Setup

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Make scripts/ importable (the notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from rag_search import (format_docs, get_config, get_retriever, hybrid_search, keyword_search,
                        show_results, similarity_search)

cfg = get_config()  # reads .env
print(f"Chat model      : {cfg.chat_model}")
print(f"Embedding model : {cfg.embedding_model}")
print(f"Chroma server   : http://{cfg.chroma_host}:{cfg.chroma_port}  (tenant: {cfg.chroma_tenant})")
print(f"Databases       : {cfg.databases}  collection: {cfg.collection}")

Chat model      : gpt-5.5
Embedding model : text-embedding-3-small
Chroma server   : http://localhost:8000  (tenant: default_tenant)
Databases       : {'networking': 'networking_kb', 'meetings': 'meetings_kb'}  collection: kb_articles


## 1. Retrieval building blocks

The same question goes through all three retrieval modes. Each function returns `[(Document, score), ...]`, best match first.

| Mode | Score | Good at |
|---|---|---|
| `similarity_search` | relevance 0–1 (higher is better) | meaning and paraphrases |
| `keyword_search` (BM25) | BM25 (unbounded) | exact codes and terms such as `NCS-SLA-2001` |
| `hybrid_search` | RRF = Σ weight / (60 + rank) | both, fused by rank |

In [2]:
question = "What does alarm NCS-SLA-2001 mean and when does it escalate to Sev1?"

show_results(similarity_search(question, "networking"), f"Similarity | {question}")
show_results(keyword_search(question, "networking"), f"Keyword (BM25) | {question}")
show_results(hybrid_search(question, "networking"), f"Hybrid (BM25 + vector) | {question}")

Similarity | What does alarm NCS-SLA-2001 mean and when does it escalate to Sev1?                                  
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.4376 │ KB-NET-1008 │ text  │ current    │ chunk 6          │ NCS-SLA-2002 is a restoration event.      │
│    │          │             │       │            │                  │ Always review the preceding NCS-SLA-2001  │
│    │          │             │       │            │                  │ duration and whether the alarm            │
│    │          │             │       │            │                  │ auto-escala...                            │
│  2 │   0.4302 │ KB-NET-1006 │ text  │ current    │ chunk 7          │ Escalate to the transport provider if BFD │
│    │          │             │       │            │                  │ flaps correlate with packet loss,         │
│    │          │             │       │            │                  │ CRC/input errors on the access handoff,   │
│    │          │             │       │            │                  │ carr...                                   │
│  3 │   0.4217 │ KB-NET-1003 │ table │ current    │ Table 1, section │ The table defines NetCore SD-WAN 5.0 SLA  │
│    │          │             │       │            │ 'Policy          │ breach handling, escalation timing,       │
│    │          │             │       │            │ Statements'      │ automated failover, and incident          │
│    │          │             │       │            │                  │ severity. ...                             │
│  4 │   0.4126 │ KB-NET-1005 │ text  │ current    │ chunk 1          │ Troubleshooting Alarm NCS-SLA-2001 "SLA   │
│    │          │             │       │            │                  │ Violation Detected" KB-NET-1005 | NetCore │
│    │          │             │       │            │                  │ SD-WAN 5.0 | Last updated: 2026-09-21 ... │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

Keyword (BM25) | What does alarm NCS-SLA-2001 mean and when does it escalate to Sev1?                              
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │  12.9169 │ KB-NET-1009 │ text  │ current    │ chunk 2          │ Q: How long does automatic failover take  │
│    │          │             │       │            │                  │ after an SLA problem starts? A: In        │
│    │          │             │       │            │                  │ NetCore SD-WAN 5.0, the app-route poll    │
│    │          │             │       │            │                  │ inter...                                  │
│  2 │  12.5885 │ KB-NET-1005 │ text  │ current    │ chunk 5          │ ncx-edge# show sdwan app-route statistics │
│    │          │             │       │            │                  │ Timestamp: 2026-11-06 09:54:31 UTC Site:  │
│    │          │             │       │            │                  │ BR-042 Device: NCX-1000 System-IP: 10...  │
│  3 │  11.2535 │ KB-NET-1009 │ text  │ current    │ chunk 1          │ SD-WAN FAQ: Transports, Failover and SLA  │
│    │          │             │       │            │                  │ Reports KB-NET-1009 | NetCore SD-WAN 5.0  │
│    │          │             │       │            │                  │ | Last updated: 2026-08-25 | Status: c... │
│  4 │  11.0620 │ KB-NET-1003 │ table │ current    │ Table 1, section │ The table defines NetCore SD-WAN 5.0 SLA  │
│    │          │             │       │            │ 'Policy          │ breach handling, escalation timing,       │
│    │          │             │       │            │ Statements'      │ automated failover, and incident          │
│    │          │             │       │            │                  │ severity. ...                             │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

Hybrid (BM25 + vector) | What does alarm NCS-SLA-2001 mean and when does it escalate to Sev1?                      
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0158 │ KB-NET-1003 │ table │ current    │ Table 1, section │ The table defines NetCore SD-WAN 5.0 SLA  │
│    │          │             │       │            │ 'Policy          │ breach handling, escalation timing,       │
│    │          │             │       │            │ Statements'      │ automated failover, and incident          │
│    │          │             │       │            │                  │ severity. ...                             │
│  2 │   0.0082 │ KB-NET-1009 │ text  │ current    │ chunk 2          │ Q: How long does automatic failover take  │
│    │          │             │       │            │                  │ after an SLA problem starts? A: In        │
│    │          │             │       │            │                  │ NetCore SD-WAN 5.0, the app-route poll    │
│    │          │             │       │            │                  │ inter...                                  │
│  3 │   0.0082 │ KB-NET-1008 │ text  │ current    │ chunk 6          │ NCS-SLA-2002 is a restoration event.      │
│    │          │             │       │            │                  │ Always review the preceding NCS-SLA-2001  │
│    │          │             │       │            │                  │ duration and whether the alarm            │
│    │          │             │       │            │                  │ auto-escala...                            │
│  4 │   0.0081 │ KB-NET-1005 │ text  │ current    │ chunk 5          │ ncx-edge# show sdwan app-route statistics │
│    │          │             │       │            │                  │ Timestamp: 2026-11-06 09:54:31 UTC Site:  │
│    │          │             │       │            │                  │ BR-042 Device: NCX-1000 System-IP: 10...  │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

In [3]:
# Hybrid results keep each chunk's rank in the individual retrievers (0 = not retrieved by that retriever)
for doc, rrf in hybrid_search(question, "networking"):
    m = doc.metadata
    print(f"{m['kb_id']:12} {m['content_type']:6} BM25 rank={m['bm25_rank']}  vector rank={m['vector_rank']}  RRF={rrf}")

KB-NET-1003  table  BM25 rank=4  vector rank=3  RRF=0.01575
KB-NET-1009  text   BM25 rank=1  vector rank=0  RRF=0.0082
KB-NET-1008  text   BM25 rank=0  vector rank=1  RRF=0.0082
KB-NET-1005  text   BM25 rank=2  vector rank=0  RRF=0.00806


## 2. Building the context

`format_docs` turns the retrieved chunks into **numbered, citable source blocks**. Each header carries the KB ID, product version, **status** (so the LLM can spot superseded content) and **location** (page, section, table).

In [4]:
retriever = get_retriever(mode="hybrid", domain="networking")   # a LangChain BaseRetriever
docs = retriever.invoke(question)
context = format_docs(docs)

print(f"{len(docs)} chunks, {len(context):,} characters of context\n")
print(context[:2500], "\n...")

4 chunks, 9,650 characters of context

[Source 1] KB-NET-1003 | SLA Breach Policy: Escalation and Auto-Failover | NetCore SD-WAN 5.0 | status: current | location: Table 1, section 'Policy Statements'
Table 1 of KB-NET-1003 "SLA Breach Policy: Escalation and Auto-Failover" (NetCore SD-WAN 5.0) | section 'Policy Statements' | caption: NetCore SD-WAN 5.0 SLA breach and escalation policy
Description: The table defines NetCore SD-WAN 5.0 SLA breach handling, escalation timing, automated failover, and incident severity. SLA threshold exceedance is monitored at 120 seconds and 240 seconds with no breach alarm, then after 3 consecutive polling intervals, approximately 6 minutes, it raises “NCS-SLA-2001 SLA Violation Detected,” auto-creates an ITSM ticket, fails over the affected application class to the best compliant transport, and assigns Sev3. Persistent breaches escalate at “> 15 minutes” to Sev2 and “> 30 minutes” to Sev1, while “>= 5 sites affected” is Sev1 with incidents grouped or rela

## 3. Prompt with system instructions

The system prompt is what turns retrieved text into a **grounded** answer:

1. Answer **only** from the context, otherwise say so.
2. **Cite** every fact as `[KB-ID, location]`.
3. **Flag superseded** sources and prefer current ones.
4. Keep commands and values verbatim.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

NOT_FOUND = "I couldn't find this in the NetCore knowledge base."

SYSTEM_PROMPT = """You are the NetCore Systems support assistant for NetCore SD-WAN, NetCore Meet and NetCore Calling.

Rules:
1. Answer ONLY from the CONTEXT below. Do not use outside knowledge.
   If the context does not contain the answer, reply exactly: "{not_found}"
   and then, in one line, say which kind of article might contain it.
2. Cite every fact inline as [KB-ID, location], using the source headers, e.g. [KB-NET-1003, Table 1] or [KB-MTG-2003, p.2].
3. Prefer sources with status "current". If the CONTEXT contains a SUPERSEDED source on the same topic
   (even if you did not use it), add a final line starting with "Note: superseded source" that names
   the old article, its release and value, and the current article that replaces it.
4. Be concise: short paragraphs or bullets. Copy CLI commands, codes and values verbatim.

CONTEXT:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
]).partial(not_found=NOT_FOUND)

# What the LLM will actually receive
for message in prompt.format_messages(context=context, question=question):
    print(f"===== {message.type.upper()} ({len(message.content):,} chars) =====")
    print(message.content[:1500] + ("\n..." if len(message.content) > 1500 else ""), "\n")

===== SYSTEM (10,527 chars) =====
You are the NetCore Systems support assistant for NetCore SD-WAN, NetCore Meet and NetCore Calling.

Rules:
1. Answer ONLY from the CONTEXT below. Do not use outside knowledge.
   If the context does not contain the answer, reply exactly: "I couldn't find this in the NetCore knowledge base."
   and then, in one line, say which kind of article might contain it.
2. Cite every fact inline as [KB-ID, location], using the source headers, e.g. [KB-NET-1003, Table 1] or [KB-MTG-2003, p.2].
3. Prefer sources with status "current". If the CONTEXT contains a SUPERSEDED source on the same topic
   (even if you did not use it), add a final line starting with "Note: superseded source" that names
   the old article, its release and value, and the current article that replaces it.
4. Be concise: short paragraphs or bullets. Copy CLI commands, codes and values verbatim.

CONTEXT:
[Source 1] KB-NET-1003 | SLA Breach Policy: Escalation and Auto-Failover | NetCore SD-WAN

## 4. Naive RAG, step by step

Four explicit steps: **retrieve → context → prompt → generate**.

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=cfg.chat_model)

question = "When does an SLA breach escalate to Sev1, and what happens before that?"

docs = get_retriever("hybrid", "networking").invoke(question)          # 1. retrieve
context = format_docs(docs)                                             # 2. build context
messages = prompt.format_messages(context=context, question=question)   # 3. build prompt
response = llm.invoke(messages)                                         # 4. generate

show_results(docs, "Retrieved sources")
print(response.content)
print("\nTokens:", response.usage_metadata)

Retrieved sources                                                                                                  
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0163 │ KB-NET-1003 │ table │ current    │ Table 1, section │ The table defines NetCore SD-WAN 5.0 SLA  │
│    │          │             │       │            │ 'Policy          │ breach handling, escalation timing,       │
│    │          │             │       │            │ Statements'      │ automated failover, and incident          │
│    │          │             │       │            │                  │ severity. ...                             │
│  2 │   0.0082 │ KB-NET-1009 │ text  │ current    │ chunk 2          │ Q: How long does automatic failover take  │
│    │          │             │       │            │                  │ after an SLA problem starts? A: In        │
│    │          │             │       │            │                  │ NetCore SD-WAN 5.0, the app-route poll    │
│    │          │             │       │            │                  │ inter...                                  │
│  3 │   0.0081 │ KB-NET-1004 │ table │ superseded │ Table 1, section │ This table defines the NetCore SD-WAN 4.2 │
│    │          │             │       │            │ 'Policy          │ SLA breach threshold and escalation       │
│    │          │             │       │            │ Statements'      │ policy used to decide when an SLA         │
│    │          │             │       │            │                  │ violatio...                               │
│  4 │   0.0079 │ KB-NET-1005 │ text  │ current    │ chunk 5          │ ncx-edge# show sdwan app-route statistics │
│    │          │             │       │            │                  │ Timestamp: 2026-11-06 09:54:31 UTC Site:  │
│    │          │             │       │            │                  │ BR-042 Device: NCX-1000 System-IP: 10...  │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

An SLA breach escalates to **Sev1** when either:

- The breach persists for **> 30 minutes** after declaration; the alarm and ITSM ticket are updated and operations channels are notified [KB-NET-1003, Table 1].
- The breach affects **>= 5 sites**; incidents are grouped or related in ITSM and operations channels are notified [KB-NET-1003, Table 1].

Before Sev1:

- After **1 polling interval / 120 seconds**, the failed interval is recorded, but no breach alarm is raised [KB-NET-1003, Table 1].
- After **2 consecutive polling intervals / 240 seconds**, monitoring continues, but no breach alarm is raised [KB-NET-1003, Table 1].
- After **3 consecutive polling intervals / approximately 6 minutes**, NetCore raises **NCS-SLA-2001 SLA Violation Detected**, auto-creates an ITSM ticket, fails over the affected application class to the best compliant transport, and assigns **Sev3** [KB-NET-1003, Table 1].
- If the breach persists for **> 15 minutes**, the alarm and ITSM ticket are updated, opera

## 5. The same flow as an LCEL chain

```
{"docs": retriever, "question": passthrough}
        └─► .assign(answer = context(docs) | prompt | llm | parser)
```

Keeping `docs` in the output lets us **show the sources** and **stream** the answer with a single retrieval.

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough


def build_rag_chain(retriever):
    generate = (
        RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
        | prompt
        | llm
        | StrOutputParser()
    )
    return RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(answer=generate)


def ask(question, domain="networking", mode="hybrid", filter=None, show_sources=True):
    # Run the RAG chain: show the retrieved sources, then stream the answer token by token
    chain = build_rag_chain(get_retriever(mode, domain, filter=filter))
    print(f"Q: {question}   [domain={domain}, mode={mode}, filter={filter}]\n")
    answer = ""
    for chunk in chain.stream(question):
        if "docs" in chunk and show_sources:
            show_results(chunk["docs"], "Sources")
        if "answer" in chunk:
            print(chunk["answer"], end="", flush=True)
            answer += chunk["answer"]
    print()
    return answer


_ = ask("How do I verify that SLA classes are applied on an NCX edge?")

Q: How do I verify that SLA classes are applied on an NCX edge?   [domain=networking, mode=hybrid, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0164 │ KB-NET-1001 │ text  │ current    │ p.1              │ NetCore Systems | KB-NET-1001 | Internal  │
│    │          │             │       │            │                  │ use only Page 1 NetCore Systems Knowledge │
│    │          │             │       │            │                  │ Base Configuring App-Aware Routing SL...  │
│  2 │   0.0081 │ KB-NET-1009 │ text  │ current    │ chunk 2          │ Q: How long does automatic failover take  │
│    │          │             │       │            │                  │ after an SLA problem starts? A: In        │
│    │          │             │       │            │                  │ NetCore SD-WAN 5.0, the app-route poll    │
│    │          │             │       │            │                  │ inter...                                  │
│  3 │   0.0081 │ KB-NET-1001 │ text  │ current    │ p.3              │ NetCore Systems | KB-NET-1001 | Internal  │
│    │          │             │       │            │                  │ use only Page 3 ncx-edge(config)# sdwan   │
│    │          │             │       │            │                  │ app-route policy AR-BRANCH-STANDARD       │
│    │          │             │       │            │                  │ ncx...                                    │
│  4 │   0.0079 │ KB-NET-1010 │ text  │ current    │ chunk 2          │ Apply the same policy object consistently │
│    │          │             │       │            │                  │ to NCX-1000 branch edges and any NCX-4000 │
│    │          │             │       │            │                  │ or NCX-9000 hub devices that partici...   │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

On the NCX edge, verify after the policy is deployed by checking SLA classes, BFD probes, control connectivity, and app-route statistics. [KB-NET-1001, p.3]

Run:

```text
show sdwan sla-class
```

Confirms the SLA classes and exact thresholds are present. [KB-NET-1001, p.3]

```text
show sdwan bfd sessions
```

Confirms tunnel probes are active. [KB-NET-1001, p.3]

```text
show sdwan control connections
```

Confirms the edge remains connected to NetCore Orchestrator. [KB-NET-1001, p.3]

```text
show sdwan app-route statistics
```

Confirms application-to-SLA-class mapping and selected transport. [KB-NET-1001, p.3]

If `NCS-SLA-2001 SLA Violation Detected` is active, compare measured latency, jitter, and loss to the configured SLA class threshold and review the affected transport. [KB-NET-1001, p.3]


## 6. Demos

### 6a. LLM alone vs LLM + RAG

NetCore is fictional, so the model **cannot** know product-specific facts such as alarm codes and timers. Without retrieval it either refuses or confidently makes something up (a hallucination). Generic numbers such as "Sev1 after 30 minutes" can be guessed correctly by luck, so this question asks for specifics.

In [8]:
question = ("In NetCore SD-WAN 5.0, which alarm code is raised for an SLA breach, "
            "and which poll interval and multiplier are used to declare it?")

print("=== LLM only (no retrieval) ===")
for token in llm.stream(question):
    print(token.content, end="", flush=True)
print("\n")

print("=== LLM + RAG ===")
_ = ask(question, domain="networking")

=== LLM only (no retrieval) ===
In NetCore SD-WAN 5.0, an SLA breach raises the **`sla-change`** alarm/event code — shown as an **SLA Violation/SLA breach** alarm.

The default declaration parameters are:

- **Poll interval:** **600,000 ms** — **10 minutes**
- **Multiplier:** **6**

So the SLA state is evaluated using the app-route/BFD SLA polling settings of **10-minute poll interval** with **multiplier 6**.

=== LLM + RAG ===
Q: In NetCore SD-WAN 5.0, which alarm code is raised for an SLA breach, and which poll interval and multiplier are used to declare it?   [domain=networking, mode=hybrid, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0160 │ KB-NET-1003 │ text  │ current    │ chunk 2          │ App-route poll interval: The interval at  │
│    │          │             │       │            │                  │ which NetCore SD-WAN evaluates path       │
│    │          │             │       │            │                  │ health for application routing. In        │
│    │          │             │       │            │                  │ release ...                               │
│  2 │   0.0082 │ KB-NET-1009 │ text  │ current    │ chunk 2          │ Q: How long does automatic failover take  │
│    │          │             │       │            │                  │ after an SLA problem starts? A: In        │
│    │          │             │       │            │                  │ NetCore SD-WAN 5.0, the app-route poll    │
│    │          │             │       │            │                  │ inter...                                  │
│  3 │   0.0081 │ KB-NET-1002 │ text  │ current    │ p.2              │ NetCore Systems | KB-NET-1002 | Internal  │
│    │          │             │       │            │                  │ use only Page 2 Application intent        │
│    │          │             │       │            │                  │ Default SLA class Evaluation behavior     │
│    │          │             │       │            │                  │ Genera...                                 │
│  4 │   0.0081 │ KB-NET-1004 │ table │ superseded │ Table 1, section │ This table defines the NetCore SD-WAN 4.2 │
│    │          │             │       │            │ 'Policy          │ SLA breach threshold and escalation       │
│    │          │             │       │            │ Statements'      │ policy used to decide when an SLA         │
│    │          │             │       │            │                  │ violatio...                               │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

In NetCore SD-WAN 5.0:

- SLA breach alarm code: `NCS-SLA-2001 SLA Violation Detected` [KB-NET-1003, chunk 2].
- App-route poll interval: `120 seconds` [KB-NET-1003, chunk 2].
- Multiplier: `3` consecutive failed intervals [KB-NET-1003, chunk 2].
- Breach declaration time: approximately `6 minutes` [KB-NET-1003, chunk 2].

Note: superseded source KB-NET-1004 for NetCore SD-WAN 4.2 used `600 seconds`, `5 consecutive failed intervals`, and approximately `50 minutes`; it is replaced by current KB-NET-1003 for NetCore SD-WAN 5.0 [KB-NET-1004, Table 1, section 'Policy Statements'] [KB-NET-1003, chunk 2].


### 6b. Similarity vs keyword vs hybrid retrieval

The question mixes an **exact error code** with **natural language**. Compare which chunks each mode retrieves, and how that changes the answer.

In [9]:
question = "Users get error MTG-JOIN-1007 when joining. What should the network team change?"

for mode in ("similarity", "keyword", "hybrid"):
    print(f"\n{'=' * 30} mode = {mode} {'=' * 30}")
    _ = ask(question, domain="meetings", mode=mode)


============================== mode = similarity ==============================
Q: Users get error MTG-JOIN-1007 when joining. What should the network team change?   [domain=meetings, mode=similarity, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.4706 │ KB-MTG-2007 │ table │ current    │ Table 1, section │ This cause-to-action matrix maps NetCore  │
│    │          │             │       │            │ 'Probable        │ Meet 5.0 meeting join symptoms to likely  │
│    │          │             │       │            │ Causes'          │ causes, primary remediation steps, and... │
│  2 │   0.4169 │ KB-MTG-2007 │ text  │ current    │ chunk 6          │ # Validate after change ncmeetdiag        │
│    │          │             │       │            │                  │ path-test --service meet-signalling --tcp │
│    │          │             │       │            │                  │ 443 --count 5 ncmeetdiag path-test        │
│    │          │             │       │            │                  │ --servic...                               │
│  3 │   0.4101 │ KB-MTG-2007 │ text  │ current    │ chunk 1          │ Troubleshooting Meeting Join Failures     │
│    │          │             │       │            │                  │ (MTG-JOIN-1003 / MTG-JOIN-1007)           │
│    │          │             │       │            │                  │ KB-MTG-2007 | NetCore Meet 5.0 | Last     │
│    │          │             │       │            │                  │ updated: 202...                           │
│  4 │   0.4073 │ KB-MTG-2007 │ code  │ current    │ Code block 5,    │ This escalation packet example documents  │
│    │          │             │       │            │ section          │ a NetCore Meet 5.0 join failure for error │
│    │          │             │       │            │ 'Escalation      │ MTG-JOIN-1007 Media Ports Blocked by ...  │
│    │          │             │       │            │ Criteria'        │                                           │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

The network team should allow outbound **UDP 9000** for Meet media and **TCP 5004** for media fallback, and keep **TCP 443** open for signalling. [KB-MTG-2007, Table 1]

Validate after the change with: [KB-MTG-2007, chunk 6]

```bash
ncmeetdiag path-test --service meet-signalling --tcp 443 --count 5
ncmeetdiag path-test --service meet-media --udp 9000 --tcp-fallback 5004 --count 10
```

Expected result: [KB-MTG-2007, chunk 6]

```text
Signalling TCP/443: PASS
Media UDP/9000: PASS
Media TCP/5004 fallback: PASS
Overall join readiness: PASS
```

============================== mode = keyword ==============================
Q: Users get error MTG-JOIN-1007 when joining. What should the network team change?   [domain=meetings, mode=keyword, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │  12.8857 │ KB-MTG-2009 │ text  │ current    │ chunk 3          │ Mute your phone when you are not speaking │
│    │          │             │       │            │                  │ to reduce background noise. Update or     │
│    │          │             │       │            │                  │ Cancel a Scheduled Meeting Use the        │
│    │          │             │       │            │                  │ origi...                                  │
│  2 │  12.7663 │ KB-MTG-2007 │ text  │ current    │ chunk 6          │ # Validate after change ncmeetdiag        │
│    │          │             │       │            │                  │ path-test --service meet-signalling --tcp │
│    │          │             │       │            │                  │ 443 --count 5 ncmeetdiag path-test        │
│    │          │             │       │            │                  │ --servic...                               │
│  3 │  10.5617 │ KB-MTG-2010 │ text  │ current    │ chunk 2          │ For MTG-MED-4102 Media Path Degraded      │
│    │          │             │       │            │                  │ (high packet loss), see KB-MTG-2006. For  │
│    │          │             │       │            │                  │ MTG-JOIN-1003 Meeting Locked or Host      │
│    │          │             │       │            │                  │ Lobby...                                  │
│  4 │  10.3714 │ KB-MTG-2009 │ table │ current    │ Table 1, section │ This table lists common NetCore Meet 5.0  │
│    │          │             │       │            │ 'Set Meeting     │ meeting options that can be configured    │
│    │          │             │       │            │ Options Before   │ before sending a calendar invite. It is   │
│    │          │             │       │            │ Sending the      │ ...                                       │
│    │          │             │       │            │ Invite'          │                                           │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

The network team should update firewall policy to allow:

- `TCP 443` for signalling [KB-MTG-2010, chunk 2]
- `UDP 9000` for preferred media [KB-MTG-2010, chunk 2]
- `TCP 5004` for media fallback [KB-MTG-2010, chunk 2]

`UDP 9000` should be permitted whenever possible because it is the preferred media path for NetCore Meet [KB-MTG-2010, chunk 2]. NetCore Meet can fall back to `TCP 5004` if `UDP 9000` is blocked, provided `TCP 5004` is allowed [KB-MTG-2010, chunk 2].

Validate after the change:

```bash
ncmeetdiag path-test --service meet-signalling --tcp 443 --count 5
ncmeetdiag path-test --service meet-media --udp 9000 --tcp-fallback 5004 --count 10
```  
[KB-MTG-2007, chunk 6]

Expected result:

```text
Signalling TCP/443: PASS
Media UDP/9000: PASS
Media TCP/5004 fallback: PASS
Overall join readiness: PASS
```  
[KB-MTG-2007, chunk 6]

If `MTG-JOIN-1007` is reproduced from two or more endpoints on the same network, or if `UDP 9000` and `TCP 5004` fail from the endpoint, escalate to t

Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0161 │ KB-MTG-2007 │ text  │ current    │ chunk 6          │ # Validate after change ncmeetdiag        │
│    │          │             │       │            │                  │ path-test --service meet-signalling --tcp │
│    │          │             │       │            │                  │ 443 --count 5 ncmeetdiag path-test        │
│    │          │             │       │            │                  │ --servic...                               │
│  2 │   0.0082 │ KB-MTG-2009 │ text  │ current    │ chunk 3          │ Mute your phone when you are not speaking │
│    │          │             │       │            │                  │ to reduce background noise. Update or     │
│    │          │             │       │            │                  │ Cancel a Scheduled Meeting Use the        │
│    │          │             │       │            │                  │ origi...                                  │
│  3 │   0.0082 │ KB-MTG-2007 │ table │ current    │ Table 1, section │ This cause-to-action matrix maps NetCore  │
│    │          │             │       │            │ 'Probable        │ Meet 5.0 meeting join symptoms to likely  │
│    │          │             │       │            │ Causes'          │ causes, primary remediation steps, and... │
│  4 │   0.0079 │ KB-MTG-2010 │ text  │ current    │ chunk 2          │ For MTG-MED-4102 Media Path Degraded      │
│    │          │             │       │            │                  │ (high packet loss), see KB-MTG-2006. For  │
│    │          │             │       │            │                  │ MTG-JOIN-1003 Meeting Locked or Host      │
│    │          │             │       │            │                  │ Lobby...                                  │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

For **MTG-JOIN-1007**, the network team should update firewall policy to:

- Allow **TCP 443** for signalling. [KB-MTG-2010, chunk 2]
- Allow **UDP 9000** for preferred media. [KB-MTG-2010, chunk 2]
- Allow **TCP 5004** for media fallback. [KB-MTG-2010, chunk 2]
- Specifically for MTG-JOIN-1007, allow outbound **UDP 9000** and **TCP 5004**, while keeping **TCP 443** open for signalling. [KB-MTG-2007, Table 1]

Validate after the change with:

```bash
ncmeetdiag path-test --service meet-signalling --tcp 443 --count 5
ncmeetdiag path-test --service meet-media --udp 9000 --tcp-fallback 5004 --count 10
```

Expected result:

```text
Signalling TCP/443: PASS
Media UDP/9000: PASS
Media TCP/5004 fallback: PASS
Overall join readiness: PASS
```

[KB-MTG-2007, chunk 6]


### 6c. Metadata filtering: superseded vs current

`KB-MTG-2004` (release 4.2, **superseded**) says recordings are kept for 180 days. `KB-MTG-2003` (5.0, current) says 365 days.
Watch which sources are retrieved, and how the system prompt and then a `status=current` **filter** handle the conflict.

In [10]:
question = "How long are cloud meeting recordings retained?"

print("=== No filter ===")
_ = ask(question, domain="meetings", mode="similarity")

print("\n=== filter = {'status': 'current'} ===")
_ = ask(question, domain="meetings", mode="similarity", filter={"status": "current"})

=== No filter ===
Q: How long are cloud meeting recordings retained?   [domain=meetings, mode=similarity, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.5209 │ KB-MTG-2004 │ text  │ superseded │ chunk 2          │ Transcript: A text representation of      │
│    │          │             │       │            │                  │ meeting audio. Transcripts are not        │
│    │          │             │       │            │                  │ generated by NetCore Meet 4.2 cloud       │
│    │          │             │       │            │                  │ recording. P...                           │
│  2 │   0.4561 │ KB-MTG-2003 │ table │ current    │ Table 1, p.2     │ This table lists NetCore Meet 5.0 meeting │
│    │          │             │       │            │                  │ policy thresholds and escalation          │
│    │          │             │       │            │                  │ requirements for cloud recording          │
│    │          │             │       │            │                  │ retention an...                           │
│  3 │   0.4497 │ KB-MTG-2004 │ text  │ superseded │ chunk 1          │ NetCore Systems Knowledge Base Recording  │
│    │          │             │       │            │                  │ Retention Policy (Release 4.2)            │
│    │          │             │       │            │                  │ KB-MTG-2004 | NetCore Meet 4.2 | Last     │
│    │          │             │       │            │                  │ updated: 2...                             │
│  4 │   0.4357 │ KB-MTG-2004 │ table │ superseded │ Table 1, section │ This table defines the NetCore Meet 4.2   │
│    │          │             │       │            │ 'Policy          │ recording retention policy by recording   │
│    │          │             │       │            │ Statements'      │ age, system behavior, administrator       │
│    │          │             │       │            │                  │ acti...                                   │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

Cloud meeting recordings in NetCore Meet 5.0 are retained for **365 days by default**, and the retention period is configurable from **30 to 1825 days**. Values below **30 days** or above **1825 days** must be rejected and restored to a compliant value. [KB-MTG-2003, Table 1, p.2]

Note: superseded source KB-MTG-2004 for NetCore Meet 4.2 used a fixed **180-day** retention value; it is replaced by current article KB-MTG-2003 for NetCore Meet 5.0. [KB-MTG-2004, chunk 1] [KB-MTG-2003, Table 1, p.2]

=== filter = {'status': 'current'} ===
Q: How long are cloud meeting recordings retained?   [domain=meetings, mode=similarity, filter={'status': 'current'}]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.4561 │ KB-MTG-2003 │ table │ current    │ Table 1, p.2     │ This table lists NetCore Meet 5.0 meeting │
│    │          │             │       │            │                  │ policy thresholds and escalation          │
│    │          │             │       │            │                  │ requirements for cloud recording          │
│    │          │             │       │            │                  │ retention an...                           │
│  2 │   0.3711 │ KB-MTG-2003 │ text  │ current    │ p.2              │ NetCore Systems | KB-MTG-2003 | Internal  │
│    │          │             │       │            │                  │ use only Page 2  Retention period: The    │
│    │          │             │       │            │                  │ number of days a cloud recording is       │
│    │          │             │       │            │                  │ ret...                                    │
│  3 │   0.3615 │ KB-MTG-2003 │ table │ current    │ Table 3, p.4     │ This table shows the NetCore Meet meeting │
│    │          │             │       │            │                  │ policy change history for recording,      │
│    │          │             │       │            │                  │ retention, legal hold, transcripts,       │
│    │          │             │       │            │                  │ lobby...                                  │
│  4 │   0.3372 │ KB-MTG-2003 │ code  │ current    │ Code block 1,    │ This NetCore Control Hub audit example    │
│    │          │             │       │            │ p.3              │ records the Organization Default meeting  │
│    │          │             │       │            │                  │ policy settings for NetCore Meet 5.0 at   │
│    │          │             │       │            │                  │ ...                                       │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

Cloud meeting recordings are retained for **365 days by default** in NetCore Meet 5.0. Administrators can configure retention from **30 to 1825 days**. [KB-MTG-2003, p.2]

If a recording is under **legal hold**, it is **not deleted when retention expires**; the recording and transcript are preserved until the hold is removed. [KB-MTG-2003, Table 1, p.2]


### 6d. Out-of-scope question

The KB lists licence tiers but **no prices**. A grounded assistant should say it doesn't know instead of inventing a number.

In [11]:
_ = ask("What is the list price of a NetCore Meet Enterprise licence per user per month?", domain="meetings")

Q: What is the list price of a NetCore Meet Enterprise licence per user per month?   [domain=meetings, mode=hybrid, filter=None]



Sources                                                                                                            
┏━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃    Score ┃ KB ID       ┃ Type  ┃ Status     ┃ Location         ┃ Snippet                                   ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │   0.0161 │ KB-MTG-2010 │ text  │ current    │ chunk 2          │ For MTG-MED-4102 Media Path Degraded      │
│    │          │             │       │            │                  │ (high packet loss), see KB-MTG-2006. For  │
│    │          │             │       │            │                  │ MTG-JOIN-1003 Meeting Locked or Host      │
│    │          │             │       │            │                  │ Lobby...                                  │
│  2 │   0.0159 │ KB-MTG-2010 │ text  │ current    │ chunk 3          │ Q: What changed from NetCore Meet 4.2 to  │
│    │          │             │       │            │                  │ 5.0 for recordings and retention? A: In   │
│    │          │             │       │            │                  │ release 5.0, cloud recordings are         │
│    │          │             │       │            │                  │ retai...                                  │
│  3 │   0.0082 │ KB-MTG-2010 │ table │ current    │ Table 1, section │ This quick reference table summarizes     │
│    │          │             │       │            │ 'Overview'       │ NetCore Meet 5.0 licensing limits and     │
│    │          │             │       │            │                  │ network requirements for planning         │
│    │          │             │       │            │                  │ deployment...                             │
│  4 │   0.0081 │ KB-MTG-2010 │ text  │ current    │ chunk 1          │ FAQ: Licensing, Limits and Network        │
│    │          │             │       │            │                  │ Requirements KB-MTG-2010 | NetCore Meet   │
│    │          │             │       │            │                  │ 5.0 | Last updated: 2026-08-11 | Status:  │
│    │          │             │       │            │                  │ curr...                                   │
└────┴──────────┴─────────────┴───────┴────────────┴──────────────────┴───────────────────────────────────────────┘

I couldn't find this in the NetCore knowledge base.
A pricing or licensing SKU article might contain it.


## 7. Takeaways: where Naive RAG falls short

| Limitation | Seen in this notebook | Fixed later by (Advanced / Agentic RAG) |
|---|---|---|
| Retrieval quality caps the answer quality | 6b: different modes return different sources | Hybrid by default, **re-ranking** |
| The question is used as-is | Codes vs natural language | **Query rewriting** / multi-query / HyDE |
| The caller must pick the domain | `domain=` argument | **Query routing** across vector DBs |
| Freshness relies on the prompt or a manual filter | 6c: superseded 4.2 content | Automatic metadata filters, self-query |
| Fixed `k`, one shot, no self-check | Always 4 chunks, one LLM call | Adaptive retrieval, **agentic** loops, evaluation |